In [107]:
# TIME SERIES

In [108]:
# # resample() resample() is like groupby() but for time — it buckets your data into time windows and aggregates.
# resample() is specifically a time-based method. It only works when your index is a datetime.
# resample("h")   # bucket by hour
# resample("D")   # bucket by day
# resample("W")   # bucket by week
# resample("ME")  # bucket by month end
# resample("2h")   # every 2 hours
# resample("15min") # every 15 minutes
# resample("3D")   # every 3 days
# Without a datetime index — resample() throws an error. 
# That's why set_index("timestamp") is always the first step.

In [109]:
import pandas as pd
import numpy as np

df_metrics = pd.read_csv("server_metrics.csv")
df_tickets = pd.read_csv("incidents.csv")
df_logs = pd.read_csv("app_logs.csv")
print(df_metrics.dtypes)
df_metrics["timestamp"] = pd.to_datetime(df_metrics["timestamp"])
print(df_metrics.dtypes)
print(df_metrics)

timestamp       object
server_id       object
cpu_pct        float64
memory_pct     float64
response_ms    float64
disk_pct       float64
status          object
dtype: object
timestamp      datetime64[ns]
server_id              object
cpu_pct               float64
memory_pct            float64
response_ms           float64
disk_pct              float64
status                 object
dtype: object
              timestamp server_id  cpu_pct  memory_pct  response_ms  disk_pct  \
0   2026-01-01 00:00:00    srv-01     49.6        94.6        891.8      68.9   
1   2026-01-01 00:00:00    srv-02     32.3        33.9       1046.1      69.1   
2   2026-01-01 00:00:00    srv-03     21.6        96.0       1007.3      43.8   
3   2026-01-01 00:00:00    srv-04     34.5        50.7        653.5      58.1   
4   2026-01-01 00:00:00    srv-05     68.3        39.5        386.0      53.8   
..                  ...       ...      ...         ...          ...       ...   
520 2026-01-01 04:00:00    srv-01 

In [110]:
# Hourly average CPU across fleet
hourly_cpu = df_metrics.set_index("timestamp").resample("h")["cpu_pct"].mean().round(2)
print(hourly_cpu.head(10))
# gives you fleet-wide average per hour — all 5 servers collapsed into one number per hour.

timestamp
2026-01-01 00:00:00    41.26
2026-01-01 01:00:00    67.16
2026-01-01 02:00:00    76.80
2026-01-01 03:00:00    59.98
2026-01-01 04:00:00    55.54
2026-01-01 05:00:00    54.22
2026-01-01 06:00:00    67.90
2026-01-01 07:00:00    45.56
2026-01-01 08:00:00    53.08
2026-01-01 09:00:00    77.36
Freq: h, Name: cpu_pct, dtype: float64


In [111]:
# Daily max response time per server
daily_response = df_metrics.set_index("timestamp").groupby("server_id", observed=True).resample("D") \
["response_ms"].max().round(2)
print(daily_response.head(5))

server_id  timestamp 
srv-01     2026-01-01    1130.4
           2026-01-02    1129.9
           2026-01-03    1196.2
           2026-01-04    1178.5
           2026-01-05    1149.3
Name: response_ms, dtype: float64


In [112]:
# Daily ticket count
df_tickets["created_at"] = pd.to_datetime(df_tickets["created_at"])
daily_tickets = df_tickets.set_index("created_at").resample("D")["ticket_id"].count()
print(daily_tickets)

created_at
2026-01-01    1
2026-01-02    2
2026-01-03    3
2026-01-04    6
2026-01-05    2
             ..
2026-04-06    1
2026-04-07    2
2026-04-08    3
2026-04-09    2
2026-04-10    3
Freq: D, Name: ticket_id, Length: 100, dtype: int64


In [113]:
# shift() - compare current vs previous reading

In [114]:
# How much did CPU change from previous reading ?
df_metrics_ts = df_metrics.set_index("timestamp").sort_index()
df_metrics_ts["cpu_prev"] = df_metrics_ts.groupby("server_id", observed=True)["cpu_pct"].shift(1)

In [115]:
df_metrics_ts["cpu_change"] = df_metrics_ts["cpu_pct"] - df_metrics_ts["cpu_prev"]
print(df_metrics_ts[["server_id", "cpu_pct", "cpu_prev", "cpu_change"]].head(10))

           server_id  cpu_pct  cpu_prev  cpu_change
timestamp                                          
2026-01-01    srv-01     49.6       NaN         NaN
2026-01-01    srv-03     21.6       NaN         NaN
2026-01-01    srv-01     49.6      49.6         0.0
2026-01-01    srv-02     32.3       NaN         NaN
2026-01-01    srv-05     68.3       NaN         NaN
2026-01-01    srv-04     34.5       NaN         NaN
2026-01-01    srv-02     32.3      32.3         0.0
2026-01-01    srv-03     21.6      21.6         0.0
2026-01-01    srv-04     34.5      34.5         0.0
2026-01-01    srv-05     68.3      68.3         0.0


In [116]:
# diff() - Rate of change
# diff() is shorthand for value - shift(1)
df_metrics_ts["cpu_diff"] = df_metrics_ts.groupby("server_id", observed=True)["cpu_pct"].diff()
print(df_metrics_ts[["server_id", "cpu_pct", "cpu_diff"]].head(10))
# diff() > 20 in one interval = rapid spike worth alerting on

           server_id  cpu_pct  cpu_diff
timestamp                              
2026-01-01    srv-01     49.6       NaN
2026-01-01    srv-03     21.6       NaN
2026-01-01    srv-01     49.6       0.0
2026-01-01    srv-02     32.3       NaN
2026-01-01    srv-05     68.3       NaN
2026-01-01    srv-04     34.5       NaN
2026-01-01    srv-02     32.3       0.0
2026-01-01    srv-03     21.6       0.0
2026-01-01    srv-04     34.5       0.0
2026-01-01    srv-05     68.3       0.0


In [117]:
# rolling() - Moving averages
# 3-period rolling average CPU per server
df_metrics_ts["cpu_rolling_3"] = df_metrics_ts.groupby("server_id", observed=True)["cpu_pct"].\
    transform(lambda x: x.rolling(window=3).mean())
print(df_metrics_ts[["server_id", "cpu_pct", "cpu_rolling_3"]].head(15))

                    server_id  cpu_pct  cpu_rolling_3
timestamp                                            
2026-01-01 00:00:00    srv-01     49.6            NaN
2026-01-01 00:00:00    srv-03     21.6            NaN
2026-01-01 00:00:00    srv-01     49.6            NaN
2026-01-01 00:00:00    srv-02     32.3            NaN
2026-01-01 00:00:00    srv-05     68.3            NaN
2026-01-01 00:00:00    srv-04     34.5            NaN
2026-01-01 00:00:00    srv-02     32.3            NaN
2026-01-01 00:00:00    srv-03     21.6            NaN
2026-01-01 00:00:00    srv-04     34.5            NaN
2026-01-01 00:00:00    srv-05     68.3            NaN
2026-01-01 01:00:00    srv-01     82.0      60.400000
2026-01-01 01:00:00    srv-02     68.0      44.200000
2026-01-01 01:00:00    srv-03     83.9      42.366667
2026-01-01 01:00:00    srv-04     29.6      32.866667
2026-01-01 01:00:00    srv-05     72.3      69.633333


In [118]:
# rolling average smooths out noise - compare raw vs smoothed
df_metrics_ts["response_rolling_5"] = df_metrics_ts.groupby("server_id", observed=True)\
["response_ms"].transform(lambda x: x.rolling(window=5).mean())
print(df_metrics_ts[["server_id", "response_ms", "response_rolling_5"]].head(15))

                    server_id  response_ms  response_rolling_5
timestamp                                                     
2026-01-01 00:00:00    srv-01        891.8                 NaN
2026-01-01 00:00:00    srv-03       1007.3                 NaN
2026-01-01 00:00:00    srv-01        891.8                 NaN
2026-01-01 00:00:00    srv-02       1046.1                 NaN
2026-01-01 00:00:00    srv-05        386.0                 NaN
2026-01-01 00:00:00    srv-04        653.5                 NaN
2026-01-01 00:00:00    srv-02       1046.1                 NaN
2026-01-01 00:00:00    srv-03       1007.3                 NaN
2026-01-01 00:00:00    srv-04        653.5                 NaN
2026-01-01 00:00:00    srv-05        386.0                 NaN
2026-01-01 01:00:00    srv-01        641.4                 NaN
2026-01-01 01:00:00    srv-02        124.8                 NaN
2026-01-01 01:00:00    srv-03        162.3                 NaN
2026-01-01 01:00:00    srv-04         89.5             

In [119]:
df_metrics_ts = df_metrics_ts.sort_values(
    ["server_id", "timestamp"]
)

df_metrics_ts["cpu_rolling_3w"] = (
    df_metrics_ts.groupby("server_id")["cpu_pct"]
    .transform(lambda x: x.rolling(window=3).mean())
)
print(df_metrics_ts["cpu_rolling_3w"])

timestamp
2026-01-01 00:00:00          NaN
2026-01-01 00:00:00          NaN
2026-01-01 01:00:00    60.400000
2026-01-01 01:00:00    71.200000
2026-01-01 02:00:00    86.866667
                         ...    
2026-01-04 23:00:00    63.866667
2026-01-05 00:00:00    50.466667
2026-01-05 01:00:00    41.266667
2026-01-05 02:00:00    38.166667
2026-01-05 03:00:00    46.933333
Name: cpu_rolling_3w, Length: 525, dtype: float64


In [120]:
# Time based filtering patterns

In [121]:
# Filter last 24 hours of data
latest = df_metrics["timestamp"].max()
last_24h = df_metrics[df_metrics["timestamp"] >= (latest - pd.Timedelta(hours=24))]
print(f"Last 24h rows: {len(last_24h)}")


Last 24h rows: 125


In [122]:
# Filter business hours only - 9am to 6pm
df_metrics_ts["hour"] = df_metrics_ts.index.hour
business_hours = df_metrics_ts[df_metrics_ts["hour"].between(9,18)]
print(f"Business hours rows: {len(business_hours)}")
# incidents during business hours vs off-hours have different response SLAs

Business hours rows: 200


In [123]:
# AIOPS pattern : spike detection

In [124]:
# Flag readings where CPU jumped more than 20 points in one interval
df_metrics_ts["cpu_diff"] = df_metrics_ts.groupby("server_id", observed=True)["cpu_pct"].diff()
spikes = df_metrics_ts[df_metrics_ts["cpu_diff"] > 20]
print(f"cpu spikes detected:", {len(spikes)})
print(spikes[["server_id", "cpu_pct", "cpu_diff"]].sort_values("cpu_diff", ascending=False).head(10))
# AIOPS Use - sudden jumps in CPU/memory/response are leadning indicators of incidents. 
# This feeds directly into alert rule engines.

cpu spikes detected: {147}
                    server_id  cpu_pct  cpu_diff
timestamp                                       
2026-01-01 21:00:00    srv-04     95.4      73.3
2026-01-04 20:00:00    srv-05     91.7      71.1
2026-01-02 03:00:00    srv-01     97.6      70.7
2026-01-04 07:00:00    srv-01     94.3      67.3
2026-01-01 15:00:00    srv-05     93.5      65.5
2026-01-03 20:00:00    srv-04     98.6      64.8
2026-01-04 00:00:00    srv-04     96.5      64.6
2026-01-03 09:00:00    srv-01     96.2      63.0
2026-01-01 01:00:00    srv-03     83.9      62.3
2026-01-01 09:00:00    srv-02     82.8      62.1


In [125]:
# Rule                                            #  Why
#-------                                           -------
# Always set_index("timestamp") before resample    resample requires datetime index
# Always groupby before shift/diff/rolling         Per-server calculation, not fleet-wide
# Use transform() with rolling                     Keeps original DataFrame shape
# pd.Timedelta() for relative time filters         Clean, readable time arithmetic
# Use "h" not "H"                                  pandas 2.x deprecated uppercase aliases

In [126]:
# LAB TASKS

In [127]:
# Task 1 — server_metrics
# Resample to hourly average for cpu_pct, memory_pct, response_ms across entire fleet. 
# Which hour had the highest average CPU?

In [128]:
df_metrics_ts = df_metrics.set_index("timestamp")
hourly_avg = df_metrics_ts.resample("h")[["cpu_pct", "memory_pct", "response_ms"]].mean()
max_cpu_hour = hourly_avg["cpu_pct"].idxmax()
print(max_cpu_hour)
print(hourly_avg.loc[max_cpu_hour, "cpu_pct"])

2026-01-02 04:00:00
85.58


In [129]:
# Task 2 — server_metrics
# Resample to daily max response_ms per server. 
# Which server had the highest single-day max response time and on which date?

In [130]:
df_metrics.set_index("timestamp").groupby("server_id", observed=True).resample("D")["response_ms"].max()

server_id  timestamp 
srv-01     2026-01-01    1130.4
           2026-01-02    1129.9
           2026-01-03    1196.2
           2026-01-04    1178.5
           2026-01-05    1149.3
srv-02     2026-01-01    1196.1
           2026-01-02    1182.1
           2026-01-03    1164.8
           2026-01-04    1104.1
           2026-01-05    1181.6
srv-03     2026-01-01    1143.2
           2026-01-02    1135.7
           2026-01-03    1148.9
           2026-01-04    1195.8
           2026-01-05    1043.1
srv-04     2026-01-01    1102.1
           2026-01-02    1159.8
           2026-01-03    1120.2
           2026-01-04    1084.8
           2026-01-05     926.2
srv-05     2026-01-01    1165.4
           2026-01-02    1196.4
           2026-01-03    1075.9
           2026-01-04    1162.2
           2026-01-05    1095.9
Name: response_ms, dtype: float64

In [131]:
# Task 3 — server_metrics
# Use diff() to detect CPU spikes > 30 points in one interval per server. 
# Which server has the most spikes?

In [132]:
df_metrics_ts = df_metrics.set_index("timestamp")
df_metrics_ts["cpu_diff"] = df_metrics_ts.groupby("server_id", observed=True)["cpu_pct"].diff()
spikes = df_metrics_ts[df_metrics_ts["cpu_diff"] > 30]
print(spikes[["server_id", "cpu_pct", "cpu_diff"]])
spike_counts = spikes.groupby("server_id").size()
print(spike_counts)
max_spike_server = spike_counts.idxmax()
print(max_spike_server)

# Set timestamp index
# Calculate CPU difference per server
# Detect spikes > 30
# Count spikes per server
# Server with most spikes

                    server_id  cpu_pct  cpu_diff
timestamp                                       
2026-01-01 01:00:00    srv-01     82.0      32.4
2026-01-01 01:00:00    srv-02     68.0      35.7
2026-01-01 01:00:00    srv-03     83.9      62.3
2026-01-01 02:00:00    srv-04     62.9      33.3
2026-01-01 04:00:00    srv-04     88.8      59.4
...                       ...      ...       ...
2026-01-01 01:00:00    srv-01     82.0      32.4
2026-01-01 01:00:00    srv-02     68.0      35.7
2026-01-01 01:00:00    srv-03     83.9      62.3
2026-01-01 02:00:00    srv-04     62.9      33.3
2026-01-01 04:00:00    srv-04     88.8      59.4

[106 rows x 3 columns]
server_id
srv-01    26
srv-02    18
srv-03    22
srv-04    25
srv-05    15
dtype: int64
srv-01


In [133]:
# Task 4 — server_metrics
# Add a 5-period rolling average for response_ms per server. 
# Flag rows where raw response_ms exceeds rolling average by more than 200ms — these are anomalous spikes.

In [134]:
df_metrics.groupby("server_id", observed=True)["response_ms"].transform(lambda x:x.rolling(window=5).mean())

0         NaN
1         NaN
2         NaN
3         NaN
4         NaN
        ...  
520    642.06
521    559.14
522    815.00
523    581.00
524    409.58
Name: response_ms, Length: 525, dtype: float64

In [135]:
# Task 5 — incidents
# Resample df_tickets by day on created_at. 
# Count tickets created per day. Which day had the most incidents created?

In [136]:
df_tickets["created_at"] = pd.to_datetime(df_tickets["created_at"])
df_tickets_ts = df_tickets.set_index("created_at").resample("D")["ticket_id"].count()
print(df_tickets_ts.head(10))
print(df_tickets_ts.idxmax())
print(f"highest tickets {df_tickets_ts.idxmax().date()} No of tickets {df_tickets_ts.max()}")

created_at
2026-01-01    1
2026-01-02    2
2026-01-03    3
2026-01-04    6
2026-01-05    2
2026-01-06    3
2026-01-07    1
2026-01-08    0
2026-01-09    7
2026-01-10    2
Freq: D, Name: ticket_id, dtype: int64
2026-01-09 00:00:00
highest tickets 2026-01-09 No of tickets 7


In [137]:
df_tickets.dtypes

ticket_id              object
server_id              object
category               object
priority                int64
created_at     datetime64[ns]
resolved_at            object
resolved                 bool
dtype: object

In [138]:
# Task 6 — app_logs
# Filter logs to last 48 hours of data. 
# Among those — count ERROR and CRITICAL logs per server. Which server had the most high severity logs in that window?

In [163]:
df_logs["timestamp"] = pd.to_datetime(df_logs["timestamp"])
df_logs.set_index("timestamp")
latest = df_logs["timestamp"].max()
last_48h = df_logs[df_logs["timestamp"] >= latest - pd.Timedelta(hours=48)]
print("last 48h count", len(last_48h))
high_logs = last_48h[last_48h["log_level"].isin(["ERROR","CRTICAL"])]
log_counts = (high_logs.groupby("server_id")["log_level"].count())

print("log counts:\n ", log_counts)

# Server with most high severity logs
print("Server with most high severity logs:")
print(log_counts.idxmax())

# Maximum count
print("Count:")
print(log_counts.max())

last 48h count 55
log counts:
  server_id
srv-01    1
srv-02    2
srv-03    3
srv-04    4
srv-05    2
Name: log_level, dtype: int64
Server with most high severity logs:
srv-04
Count:
4


In [158]:
df_logs.dtypes

timestamp     datetime64[ns]
server_id             object
log_level             object
message               object
error_code            object
dtype: object

In [140]:
# Task 7 — cross-dataset
# Use diff() on response_ms per server. 
# Join the top 10 largest response spikes with df_tickets on server_id. 
# Were there open P1 tickets on those servers at the time?

In [155]:
df_tickets.head(2)

,ticket_id,server_id,category,priority,created_at,resolved_at,resolved
0,INC0001,srv-04,Service Down,2,2026-03-24 02:00:00,NaN,False
1,INC0002,srv-01,CPU High,3,2026-03-19 15:00:00,2026-03-21 12:00:00,True


In [157]:
df_metrics = df_metrics.sort_values(["server_id", "timestamp"])

df_metrics["response_diff"] = df_metrics.groupby("server_id", observed=True)["response_ms"].diff()
df_metrics
top_spikes = df_metrics.nlargest(10, "response_diff")
p1_tickets = df_tickets[
    (df_tickets["priority"] == 1) &
    (df_tickets["resolved"] == False)
]
correlated = top_spikes.merge(
    p1_tickets,
    on="server_id",
    how="left",
)
print(correlated[
    [
        "server_id",
        "timestamp",
        "response_ms",
        "response_diff",
        "ticket_id",
        "priority",
        "resolved"
    ]
])

   server_id           timestamp  response_ms  response_diff ticket_id  \
0     srv-01 2026-01-03 09:00:00       1144.6         1052.4   INC0041   
1     srv-01 2026-01-03 09:00:00       1144.6         1052.4   INC0054   
2     srv-01 2026-01-03 09:00:00       1144.6         1052.4   INC0072   
3     srv-01 2026-01-03 09:00:00       1144.6         1052.4   INC0086   
4     srv-01 2026-01-03 09:00:00       1144.6         1052.4   INC0108   
5     srv-01 2026-01-03 09:00:00       1144.6         1052.4   INC0112   
6     srv-01 2026-01-03 09:00:00       1144.6         1052.4   INC0181   
7     srv-05 2026-01-01 20:00:00       1165.4         1026.0   INC0080   
8     srv-05 2026-01-01 20:00:00       1165.4         1026.0   INC0099   
9     srv-05 2026-01-01 20:00:00       1165.4         1026.0   INC0116   
10    srv-05 2026-01-01 20:00:00       1165.4         1026.0   INC0162   
11    srv-05 2026-01-01 20:00:00       1165.4         1026.0   INC0165   
12    srv-05 2026-01-01 20:00:00      